# Phase 7: Prior Case Retrieval - GraphProp with Adapters

## Task
Prior case retrieval: for each query case, find the earlier cases it cites from a pool of candidate cases. The 7 best candidates are returned per query and scored with Micro-F1@7.

Model: the pre-trained GraphProp model with bottleneck adapters after every feed-forward block; everything else is frozen.

## Input
- Phase-7 data (queries and candidates for train / dev / test); downloaded from Hugging Face if not attached (needs HF_TOKEN).
- GraphProp checkpoint and tokenizer (found automatically).

## Flow of the notebook (from input to output)
1. Write the model code file.
2. Import libraries and set up the file finder.
3. Load the GraphProp model and set the adapter settings.
4. Load the queries and candidate cases of the train, dev and test splits.
5. Compute BM25 and TF-IDF scores between every query and every candidate.
6. Define the Micro-F1@7 metric and the score mixing (z-scores, hubness correction, balancing).
7. Define the fine-tuning methods (LoRMA, adapters, full) with memory-saving blocks.
8. Define how documents become token chunks and padded batches.
9. Define how documents become vectors (mean over tokens and chunks) and cosine similarity scores.
10. Define the training loss: contrastive (each query must pick its cited case among hard negatives), plus distillation if a teacher is given.
11. Define the training loop (cosine learning rate, dev check every epoch, keep the best epoch).
12. Tokenize everything, build positive pairs and hard negatives from BM25 / TF-IDF.
13. Fine-tune the model.
14. Compute and save the model scores for all splits.
15. Choose the model weight, BM25 weight and post-processing on dev and write the 4 best submissions.
16. Zip all outputs.

## Output
- 4 submission zips (dev score and settings in the file name).
- Saved score matrices (a file ending in _scores.npz) and a results JSON.
- One zip with all outputs.

In [1]:
%%writefile arch_graphprop.py
# GraphProp model code (saved as arch_graphprop.py)
import os, json, zipfile, tempfile, shutil, re as _re
from collections import Counter, defaultdict as _defaultdict

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

ARCH = "graphprop"

class TokenPosEmbed(nn.Module):
    # It takes vocab_size (vocabulary size), dim (model size) and max_len (max length) as input, creates the token and position embedding tables and returns nothing.
    def __init__(self, vocab_size, dim, max_len=1024):
        super().__init__()
        self.tok = nn.Embedding(vocab_size, dim)
        self.pos = nn.Embedding(max_len, dim)

    # It takes input_ids (token ids) and pos_offset (position offset) as input, adds token and position embeddings for the input and returns embedding vectors.
    def forward(self, input_ids, pos_offset=0):
        B, T = input_ids.shape
        positions = (torch.arange(T, device=input_ids.device) + pos_offset).unsqueeze(0).expand(B, -1)
        return self.tok(input_ids) + self.pos(positions)

class FeedForward(nn.Module):

    # It takes dim (model size), mult (size multiplier) and dropout (dropout rate) as input, creates the feed-forward layer and returns nothing.
    def __init__(self, dim, mult=4, dropout=0.1):
        super().__init__()
        self.norm = nn.LayerNorm(dim)
        self.net = nn.Sequential(
            nn.Linear(dim, dim * mult),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(dim * mult, dim),
            nn.Dropout(dropout),
        )

    # It takes x (input tensor) as input, runs the feed-forward layer and returns output vectors.
    def forward(self, x):
        return x + self.net(self.norm(x))

class GraphPropLayer(nn.Module):

    # It takes dim (model size), window (window size) and edge_dim (edge size) as input, creates the graph message-passing layer and returns nothing.
    def __init__(self, dim, window=16, edge_dim=64):
        super().__init__()
        self.window = window
        self.norm = nn.LayerNorm(dim)
        self.edge_key = nn.Linear(dim, edge_dim)
        self.edge_mlp = nn.Sequential(nn.Linear(edge_dim * 2, edge_dim), nn.GELU(), nn.Linear(edge_dim, 1))
        self.value_proj = nn.Linear(dim, dim)
        self.rel_emb = nn.Embedding(window, dim)
        self.out_proj = nn.Linear(dim, dim)

    # It takes x (input tensor) as input, passes messages from nearby previous tokens to each token and returns output vectors.
    def forward(self, x):

        B, T, D = x.shape
        W = self.window
        h = self.norm(x)

        K = self.edge_key(h)
        e = K.size(-1)
        pad_rows = self.edge_key.bias.to(K.dtype).expand(B, W - 1, -1)
        K_all = torch.cat([pad_rows, K], dim=1)
        w1, b1 = self.edge_mlp[0].weight, self.edge_mlp[0].bias
        A = F.linear(K, w1[:, :e], b1)
        Bn = F.linear(K_all, w1[:, e:])
        z = A.unsqueeze(2) + Bn.unfold(1, W, 1).permute(0, 1, 3, 2)
        edge_w = torch.sigmoid(self.edge_mlp[2](self.edge_mlp[1](z))).squeeze(-1)

        wv, bv = self.value_proj.weight, self.value_proj.bias
        V = F.pad(F.linear(h, wv), (0, 0, W - 1, 0))
        R = F.linear(self.rel_emb.weight, wv, bv)
        t_idx = torch.arange(T, device=x.device).unsqueeze(1) + torch.arange(W, device=x.device).unsqueeze(0)
        M = torch.zeros(B, T, T + W - 1, dtype=edge_w.dtype, device=x.device)
        M.scatter_(2, t_idx.unsqueeze(0).expand(B, -1, -1), edge_w)
        num = torch.bmm(M, V.to(M.dtype)) + torch.matmul(edge_w, R.to(edge_w.dtype))

        denom = edge_w.sum(-1, keepdim=True).clamp_min(1e-6)
        agg = num / denom
        out = self.out_proj(agg)
        return x + out

    # It takes h (normalised hidden vectors), valid (mask of valid neighbours) and drop_self (skip the token itself) as input, computes the message sums used by the layer and returns (message sum, edge weight sum).
    def _num_den(self, h, valid=None, drop_self=False):

        B, T, D = h.shape
        W = self.window
        K = self.edge_key(h)
        e = K.size(-1)
        pad_rows = self.edge_key.bias.to(K.dtype).expand(B, W - 1, -1)
        K_all = torch.cat([pad_rows, K], dim=1)
        w1, b1 = self.edge_mlp[0].weight, self.edge_mlp[0].bias
        A = F.linear(K, w1[:, :e], b1)
        Bn = F.linear(K_all, w1[:, e:])
        z = A.unsqueeze(2) + Bn.unfold(1, W, 1).permute(0, 1, 3, 2)
        edge_w = torch.sigmoid(self.edge_mlp[2](self.edge_mlp[1](z))).squeeze(-1)
        if valid is not None:
            vpad = torch.cat([torch.ones(B, W - 1, dtype=edge_w.dtype, device=h.device),
                              valid.to(edge_w.dtype)], dim=1)
            edge_w = edge_w * vpad.unfold(1, W, 1)
        if drop_self:
            edge_w = torch.cat([edge_w[..., :W - 1], torch.zeros_like(edge_w[..., :1])], dim=-1)
        wv, bv = self.value_proj.weight, self.value_proj.bias
        V = F.pad(F.linear(h, wv), (0, 0, W - 1, 0))
        R = F.linear(self.rel_emb.weight, wv, bv)
        t_idx = torch.arange(T, device=h.device).unsqueeze(1) + torch.arange(W, device=h.device).unsqueeze(0)
        M = torch.zeros(B, T, T + W - 1, dtype=edge_w.dtype, device=h.device)
        M.scatter_(2, t_idx.unsqueeze(0).expand(B, -1, -1), edge_w)
        num = torch.bmm(M, V.to(M.dtype)) + torch.matmul(edge_w, R.to(edge_w.dtype))
        return num, edge_w.sum(-1, keepdim=True)

    # It takes x (input tensor) and valid (mask of valid neighbours) as input, passes messages from tokens on both sides and returns output vectors.
    def forward_bidirectional(self, x, valid=None):

        h = self.norm(x)
        nc, dc = self._num_den(h, valid)
        vr = valid.flip(1) if valid is not None else None
        nr, dr = self._num_den(h.flip(1), vr, drop_self=True)
        num = nc + nr.flip(1)
        den = (dc + dr.flip(1)).clamp_min(1e-6)
        return x + self.out_proj(num / den)

class LegalGraphPropLM(nn.Module):
    # It takes vocab_size (vocabulary size), dim (model size), n_layers (number of layers), window (window size), edge_dim (edge size), max_len (max length), dropout (dropout rate) and tie (tie embeddings) as input, builds the full GraphProp language model and returns nothing.
    def __init__(self, vocab_size, dim=512, n_layers=6, window=16, edge_dim=64, max_len=1024,
                 dropout=0.1, tie=True):
        super().__init__()
        self.embed = TokenPosEmbed(vocab_size, dim, max_len)
        self.drop = nn.Dropout(dropout)
        layers = []
        for _ in range(n_layers):
            layers.append(GraphPropLayer(dim, window, edge_dim))
            layers.append(FeedForward(dim, dropout=dropout))
        self.blocks = nn.ModuleList(layers)
        self.norm_f = nn.LayerNorm(dim)
        self.lm_head = nn.Linear(dim, vocab_size, bias=False)
        if tie:
            self.lm_head.weight = self.embed.tok.weight

        self.apply(self._init_weights)
        residual_scale = (2 * n_layers) ** -0.5
        for blk in self.blocks:
            if isinstance(blk, GraphPropLayer):
                nn.init.normal_(blk.out_proj.weight, mean=0.0, std=0.02 * residual_scale)
            elif isinstance(blk, FeedForward):
                nn.init.normal_(blk.net[-2].weight, mean=0.0, std=0.02 * residual_scale)

    # It sets the starting values of the weights and returns nothing.
    @staticmethod
    def _init_weights(module):
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)

    # It takes input_ids (token ids), pos_offset (position offset), targets (target token ids), last_idx (last token positions), pad_id (padding id) and label_smoothing (label smoothing) as input, runs the model and returns next-token scores.
    def forward(self, input_ids, pos_offset=0, targets=None, last_idx=None, pad_id=None, label_smoothing=0.0):
        x = self.drop(self.embed(input_ids, pos_offset=pos_offset))
        for blk in self.blocks:
            x = blk(x)
        h = self.norm_f(x)
        if last_idx is not None:
            rows = torch.arange(h.size(0), device=h.device)
            return self.lm_head(h[rows, last_idx])
        logits = self.lm_head(h)
        if targets is None:
            return logits
        loss_sum = F.cross_entropy(logits.reshape(-1, logits.size(-1)), targets.reshape(-1),
                                   ignore_index=pad_id, reduction="sum", label_smoothing=label_smoothing)
        n_tok = (targets != pad_id).sum()
        return loss_sum.reshape(1), n_tok.reshape(1)

# It maps every byte to a printable character for the byte-level tokenizer and returns a dictionary from byte value to character.
def _bytes_to_unicode():
    bs = list(range(ord("!"), ord("~") + 1)) + list(range(ord("\xa1"), ord("\xac") + 1)) + \
         list(range(ord("\xae"), ord("\xff") + 1))
    cs = bs[:]
    n = 0
    for b in range(256):
        if b not in bs:
            bs.append(b)
            cs.append(256 + n)
            n += 1
    return dict(zip(bs, [chr(c) for c in cs]))

_BYTE_ENCODER = _bytes_to_unicode()
_BYTE_DECODER = {v: k for k, v in _BYTE_ENCODER.items()}
_PRETOK_RE = _re.compile(r"\'s|\'t|\'re|\'ve|\'m|\'ll|\'d| ?[A-Za-z]+| ?[0-9]+| ?[^\sA-Za-z0-9]+|\s+")

class LegalByteBPETokenizer:

    PAD_TOKEN = "<pad>"
    EOS_TOKEN = "<|endoftext|>"

    # It creates an empty tokenizer and returns nothing.
    def __init__(self):
        self.vocab = {}
        self.id_to_token = {}
        self.merge_rank = {}
        self.pad_token_id = None
        self.eos_token_id = None
        self.padding_side = "right"
        self._cache = {}

    # It returns the vocabulary size.
    def __len__(self):
        return len(self.vocab)

    # It takes texts, target_vocab_size (target vocabulary size), sample_chars_per_doc (characters used per document) and verbose (print progress) as input, learns the BPE merges and vocabulary from the training texts and returns the trained tokenizer.
    def train(self, texts, target_vocab_size, sample_chars_per_doc=None, verbose=True):
        import time as _time
        t0 = _time.time()
        word_freq = Counter()
        for text in texts:
            if sample_chars_per_doc is not None:
                text = text[:sample_chars_per_doc]
            for m in _PRETOK_RE.finditer(text):
                sym = tuple(_BYTE_ENCODER[byte] for byte in m.group().encode("utf-8"))
                word_freq[sym] += 1

        self.vocab = {self.PAD_TOKEN: 0, self.EOS_TOKEN: 1}
        for i in range(256):
            self.vocab[_BYTE_ENCODER[i]] = len(self.vocab)
        self.pad_token_id = self.vocab[self.PAD_TOKEN]
        self.eos_token_id = self.vocab[self.EOS_TOKEN]

        words = {i: list(w) for i, w in enumerate(word_freq)}
        freqs = {i: c for i, (w, c) in enumerate(word_freq.items())}
        pair_counts = Counter()
        pair_to_words = _defaultdict(set)
        for wid, syms in words.items():
            f = freqs[wid]
            for i in range(len(syms) - 1):
                p = (syms[i], syms[i + 1])
                pair_counts[p] += f
                pair_to_words[p].add(wid)

        n_merges_needed = target_vocab_size - len(self.vocab)
        n_merges_done = 0
        while n_merges_done < n_merges_needed and pair_counts:
            max_freq = max(pair_counts.values())
            best = min(p for p, c in pair_counts.items() if c == max_freq)
            new_symbol = best[0] + best[1]
            self.merge_rank[best] = n_merges_done
            self.vocab[new_symbol] = len(self.vocab)
            for wid in list(pair_to_words.get(best, ())):
                syms, f = words[wid], freqs[wid]
                for i in range(len(syms) - 1):
                    p = (syms[i], syms[i + 1])
                    pair_counts[p] -= f
                    if pair_counts[p] <= 0:
                        pair_counts.pop(p, None)
                    pair_to_words[p].discard(wid)
                merged, i = [], 0
                while i < len(syms):
                    if i < len(syms) - 1 and syms[i] == best[0] and syms[i + 1] == best[1]:
                        merged.append(new_symbol); i += 2
                    else:
                        merged.append(syms[i]); i += 1
                words[wid] = merged
                for i in range(len(merged) - 1):
                    p = (merged[i], merged[i + 1])
                    pair_counts[p] += f
                    pair_to_words[p].add(wid)
            pair_counts.pop(best, None)
            pair_to_words.pop(best, None)
            n_merges_done += 1

        self.id_to_token = {v: k for k, v in self.vocab.items()}
        if verbose:
            print(f"    BPE training: {n_merges_done} merges learned (target {n_merges_needed}) in "
                  f"{_time.time() - t0:.1f}s over {len(word_freq)} unique pretoken types "
                  f"-> final vocab size = {len(self.vocab)}")
        return self

    # It takes symbol_tuple (word as symbols) as input, applies the learned merges to one word and returns the word split into tokens (as a string).
    def _bpe(self, symbol_tuple):
        cached = self._cache.get(symbol_tuple)
        if cached is not None:
            return cached
        word = list(symbol_tuple)
        while len(word) > 1:
            best_rank, best_i = None, -1
            for i in range(len(word) - 1):
                r = self.merge_rank.get((word[i], word[i + 1]))
                if r is not None and (best_rank is None or r < best_rank):
                    best_rank, best_i = r, i
            if best_rank is None:
                break
            word = word[:best_i] + [word[best_i] + word[best_i + 1]] + word[best_i + 2:]
        self._cache[symbol_tuple] = word
        return word

    # It takes text as input, converts text into a list of token ids and returns a list of token ids.
    def encode(self, text):
        ids = []
        for m in _PRETOK_RE.finditer(text):
            sym = tuple(_BYTE_ENCODER[byte] for byte in m.group().encode("utf-8"))
            for piece in self._bpe(sym):
                ids.append(self.vocab[piece])
        return ids

    # It takes ids (token ids) and skip_special_tokens (drop special tokens) as input, converts token ids back into text and returns text.
    def decode(self, ids, skip_special_tokens=True):
        special = {self.pad_token_id, self.eos_token_id}
        pieces = [self.id_to_token.get(int(i), "") for i in ids
                  if not (skip_special_tokens and int(i) in special)]
        raw = bytes(_BYTE_DECODER[c] for c in "".join(pieces) if c in _BYTE_DECODER)
        return raw.decode("utf-8", errors="replace")

    # It takes path (file path) as input, saves the tokenizer to a JSON file and returns nothing.
    def save(self, path):
        data = {
            "vocab": self.vocab,
            "merges": [list(pair) for pair, _rank in
                       sorted(self.merge_rank.items(), key=lambda kv: kv[1])],
            "pad_token_id": self.pad_token_id,
            "eos_token_id": self.eos_token_id,
            "padding_side": self.padding_side,
        }
        with open(path, "w", encoding="utf-8") as f:
            json.dump(data, f)

    # It takes path (file path) as input, loads a tokenizer from a JSON file and returns a tokenizer.
    @classmethod
    def load(cls, path):
        with open(path, "r", encoding="utf-8") as f:
            data = json.load(f)
        tok = cls()
        tok.vocab = data["vocab"]
        tok.id_to_token = {int(v): k for k, v in tok.vocab.items()}
        tok.merge_rank = {tuple(pair): i for i, pair in enumerate(data["merges"])}
        tok.pad_token_id = data["pad_token_id"]
        tok.eos_token_id = data["eos_token_id"]
        tok.padding_side = data.get("padding_side", "right")
        return tok

    # It takes text, max_length (max length), truncation (cut long inputs), padding (padding option) and return_tensors (return tensors) as input, tokenizes one text or a list of texts, with optional cutting and padding and returns a dictionary with input_ids and attention_mask.
    def __call__(self, text, max_length=None, truncation=False, padding=None, return_tensors=None):
        is_batch = isinstance(text, (list, tuple))
        texts = list(text) if is_batch else [text]
        all_ids = [self.encode(t) for t in texts]
        if truncation and max_length is not None:
            all_ids = [ids[:max_length] for ids in all_ids]
        lens = [len(ids) for ids in all_ids]
        if padding == "max_length" and max_length is not None:
            pad_to = max_length
        elif padding is True or padding == "longest":
            pad_to = max(lens) if lens else 0
        else:
            pad_to = None
        if pad_to is not None:
            if self.padding_side == "right":
                attn = [[1] * L + [0] * (pad_to - L) for L in lens]
                all_ids = [ids + [self.pad_token_id] * (pad_to - len(ids)) for ids in all_ids]
            else:
                attn = [[0] * (pad_to - L) + [1] * L for L in lens]
                all_ids = [[self.pad_token_id] * (pad_to - len(ids)) + ids for ids in all_ids]
        else:
            attn = [[1] * L for L in lens]
        if return_tensors == "pt":
            return {"input_ids": torch.tensor(all_ids, dtype=torch.long),
                    "attention_mask": torch.tensor(attn, dtype=torch.long)}
        return {"input_ids": all_ids if is_batch else all_ids[0],
                "attention_mask": attn if is_batch else attn[0]}

class Tok:

    # It takes inner (inner tokenizer) as input, loads the tokenizer and returns nothing.
    def __init__(self, inner):
        self.inner = inner
        self.vocab_size = len(inner)
        self.pad_id = inner.pad_token_id
        self.eos_id = inner.eos_token_id
        self.bos_id = None

    # It takes text as input, converts text into token ids and returns a list of token ids.
    def encode(self, text):
        return self.inner.encode(text)

    # It takes ids (token ids) and skip_special_tokens (drop special tokens) as input, converts token ids back into text and returns text.
    def decode(self, ids, skip_special_tokens=True):
        return self.inner.decode(ids, skip_special_tokens)

    # It takes text as input, converts text into token ids and remember where each token is in the text and returns (token ids, character offsets).
    def encode_with_offsets(self, text):

        inner = self.inner
        ids, offs = [], []
        for m in _PRETOK_RE.finditer(text):
            s = m.group()
            b = s.encode("utf-8")
            byte2char = []
            for k, ch in enumerate(s):
                byte2char.extend([m.start() + k] * len(ch.encode("utf-8")))
            sym = tuple(_BYTE_ENCODER[x] for x in b)
            pos = 0
            for piece in inner._bpe(sym):
                n = len(piece)
                ids.append(inner.vocab[piece])
                offs.append((byte2char[pos], byte2char[pos + n - 1] + 1))
                pos += n
        return ids, offs

# It takes tok_path (tokenizer file path) as input, loads the tokenizer from its JSON file and returns a tokenizer.
def load_tokenizer(tok_path):

    return Tok(LegalByteBPETokenizer.load(tok_path))

# It takes path (file path) as input, turns an unzipped checkpoint folder back into a .pt file and returns (file path, temporary folder or None).
def _materialise_checkpoint_path(path):

    if not os.path.isdir(path):
        return path, None
    pkl = None
    for root, _d, files in os.walk(path):
        if "data.pkl" in files:
            pkl = root
            break
    if pkl is None:
        raise FileNotFoundError(f"{path} is a directory but contains no data.pkl")
    tmpdir = tempfile.mkdtemp(prefix="gp_ckpt_")
    out = os.path.join(tmpdir, "ckpt.pt")
    top = "archive"
    with zipfile.ZipFile(out, "w", compression=zipfile.ZIP_STORED) as zf:
        for root, _d, files in os.walk(pkl):
            for fn in sorted(files):
                full = os.path.join(root, fn)
                rel = os.path.relpath(full, pkl).replace(os.sep, "/")
                zf.write(full, arcname=f"{top}/{rel}")
    return out, tmpdir

# It takes obj (object) as input, gets the weights dictionary out of a saved checkpoint and returns weights dictionary.
def _extract_state_dict(obj):

    if isinstance(obj, nn.Module):
        return obj.state_dict()
    if isinstance(obj, dict):
        for k in ("best_state", "state_dict", "model_state_dict", "model_state", "model", "weights"):
            v = obj.get(k)
            if isinstance(v, nn.Module):
                return v.state_dict()
            if isinstance(v, dict) and any(torch.is_tensor(t) for t in v.values()):
                return v
        if any(torch.is_tensor(t) for t in obj.values()):
            return obj
    raise ValueError("could not find a state_dict in checkpoint")

# It takes sd (weights dictionary) as input, removes extra name prefixes from weight names and returns weights dictionary with clean names.
def _strip_prefixes(sd):
    out = {}
    for k, v in sd.items():
        while k.startswith("module."):
            k = k[len("module."):]
        out[k] = v
    return out

# It takes sd (weights dictionary) as input, works out the model settings from the saved weights and returns a dictionary of model settings.
def infer_config(sd):
    vocab, dim = sd["embed.tok.weight"].shape
    max_len = sd["embed.pos.weight"].shape[0]
    idx = {int(m.group(1)) for k in sd for m in [_re.match(r"blocks\.(\d+)\.", k)] if m}
    n_blocks = max(idx) + 1
    assert n_blocks % 2 == 0, "blocks should alternate GraphPropLayer/FeedForward"
    window = sd["blocks.0.rel_emb.weight"].shape[0]
    edge_dim = sd["blocks.0.edge_key.weight"].shape[0]
    return dict(vocab_size=vocab, dim=dim, n_layers=n_blocks // 2, window=window,
                edge_dim=edge_dim, max_len=max_len)

class Backbone:

    causal = True

    # It takes model and cfg (settings) as input, wraps the pre-trained model with simple helper methods and returns nothing.
    def __init__(self, model, cfg):
        self.model = model
        self.cfg = cfg
        self.d_model = cfg["dim"]
        self.max_len = cfg["max_len"]
        self.n_params = sum(p.numel() for p in model.parameters())
        self.bidirectional = False

    # It takes flag (on/off) as input, turns the causal mask off or on and returns nothing.
    def set_bidirectional(self, flag=True):
        self.bidirectional = bool(flag)
        self.causal = not self.bidirectional

    # It takes input_ids (token ids), attention_mask (mask of real tokens) and pos_offset (position offset) as input, returns the hidden vector of every token.
    def hidden(self, input_ids, attention_mask=None, pos_offset=0):
        m = self.model
        T = input_ids.size(1)
        assert T + pos_offset <= self.max_len, f"sequence {T}+{pos_offset} exceeds max_len={self.max_len}"
        x = m.drop(m.embed(input_ids, pos_offset=pos_offset))
        if self.bidirectional:
            valid = attention_mask.bool() if attention_mask is not None else None
            for blk in m.blocks:
                x = blk.forward_bidirectional(x, valid) if isinstance(blk, GraphPropLayer) else blk(x)
        else:
            for blk in m.blocks:
                x = blk(x)
        return m.norm_f(x)

    # It takes input_ids (token ids) and pos_offset (position offset) as input, returns next-token scores.
    def logits(self, input_ids, pos_offset=0):
        return self.model(input_ids, pos_offset=pos_offset)

    # It returns the list of model blocks.
    def block_modules(self):

        return list(self.model.blocks)

    # It returns the names of the linear layers inside the blocks.
    def linear_names(self):

        return [f"blocks.{n}" if n else "blocks"
                for n, mod in self.model.blocks.named_modules() if isinstance(mod, nn.Linear)]

# It takes ckpt_path (checkpoint path), device and dropout (dropout rate) as input, loads the pre-trained model from its checkpoint and returns the backbone ready to use.
def load_backbone(ckpt_path, device="cpu", dropout=0.1):

    path, tmp = _materialise_checkpoint_path(ckpt_path)
    try:
        obj = torch.load(path, weights_only=False, map_location="cpu")
    finally:
        if tmp:
            shutil.rmtree(tmp, ignore_errors=True)
    sd = _strip_prefixes(_extract_state_dict(obj))
    cfg = infer_config(sd)
    tied = ("lm_head.weight" not in sd) or torch.equal(sd["lm_head.weight"], sd["embed.tok.weight"])
    model = LegalGraphPropLM(cfg["vocab_size"], dim=cfg["dim"], n_layers=cfg["n_layers"],
                             window=cfg["window"], edge_dim=cfg["edge_dim"], max_len=cfg["max_len"],
                             dropout=dropout, tie=tied)
    if tied and "lm_head.weight" not in sd:
        sd = dict(sd); sd["lm_head.weight"] = sd["embed.tok.weight"]
    res = model.load_state_dict(sd, strict=False)
    if res.missing_keys or res.unexpected_keys:
        print("!" * 70)
        print("!!! STATE-DICT MISMATCH when loading GraphProp checkpoint !!!")
        print("missing:", res.missing_keys)
        print("unexpected:", res.unexpected_keys)
        print("!" * 70)
        raise RuntimeError("GraphProp checkpoint keys do not match model")
    model.to(device).eval()
    cfg["tied"] = tied
    return Backbone(model, cfg)

Writing arch_graphprop.py


In [2]:
# Import libraries and file finder helper
import os, re, json, glob, math, time, zipfile, numpy as np, torch, torch.nn as nn, torch.nn.functional as F, torch.utils.checkpoint
DEV, K, OUT = "cuda" if torch.cuda.is_available() else "cpu", 7, "/kaggle/working" if os.path.isdir("/kaggle/working") else "./out"
os.makedirs(OUT, exist_ok=True)
# It takes *patterns (any arguments) as input, finds the right input file to use and returns a path of the chosen file.
def pick(*patterns):
    for p in patterns:
        h = sorted(x for x in glob.glob(f"/kaggle/input/**/{p}", recursive=True) if os.path.isfile(x) and "trainable" not in x)
        if h: print("using", h[0]); return h[0]
    raise FileNotFoundError(f"none of {patterns} found in /kaggle/input")

In [3]:
# Load GraphProp model and set adapter settings
import arch_graphprop as A
bb = A.load_backbone(pick("legal_graphprop*.pt", "*graphprop*.pt")); tok = A.load_tokenizer(pick("tokenizer_graphprop*.json", "*tokenizer*graphprop*.json"))
pid = tok.pad_id or 0
NAME, TECH = "p7_gp_adapter", "adapter"
c = dict(lr=0.001, epochs=2, hours=4.0, bs=8, chunk=512, kinf=4, ktrain=2, tau=0.05, hard=40, r=16, alpha=32, ar=64, teacher="", beta=0.5, teach=0, seed=0)

using /kaggle/input/datasets/jaimingajera/final-pt-graphprop/legal_graphprop_random_final_random_90_10.pt
using /kaggle/input/datasets/jaimingajera/tokenizer-graphprop/tokenizer_graphprop.json


In [4]:
# Load queries and candidate cases
# It takes path (file path) as input, loads queries or candidates from a JSON file and returns a list of (id, text, relevant ids).
def read(path):
    r = json.load(open(path, encoding="utf-8"))
    if isinstance(r, dict): r = [dict(v, id=k) if isinstance(v, dict) else dict(id=k, text=v) for k, v in r.items()]
    print("fields:", list(r[0]))
    return [(str(d["id"]), " ".join(d["text"]) if isinstance(d["text"], list) else d["text"], [str(x) for x in d.get("relevant_candidates", [])]) for d in r]
# It takes split (split name) as input, loads the queries and candidates of one split and returns (queries, candidates).
def load_split(split):
    out = []
    for n in (split.lower() + ".json", split.lower() + "_candidates.json"):
        f = [x for x in glob.glob(f"/kaggle/input/**/{n}", recursive=True) if "phase_7" in x.lower()]
        if not f:
            from huggingface_hub import hf_hub_download, list_repo_files, login
            try:
                from kaggle_secrets import UserSecretsClient; login(UserSecretsClient().get_secret("HF_TOKEN"))
            except Exception: login(os.environ.get("HF_TOKEN"))
            repo = "Exploration-Lab/CS-781-Capstone"; f = [hf_hub_download(repo, x, repo_type="dataset") for x in list_repo_files(repo, repo_type="dataset") if x.lower().endswith("/" + n) and "phase_7" in x.lower()]
        out.append(read(f[0]))
    return out
D = {s: load_split(s) for s in ("Train", "Dev", "Test")}
print({s: (len(q), len(c_)) for s, (q, c_) in D.items()}, "(queries, candidates)")

Phase_7_PCR/Train/train.json: reconstructing file:   0%|          |  0.00B / 33.2MB            

Phase_7_PCR/Train/train.json: downloading bytes:           |  0.00B            

fields: ['text', 'relevant_candidates', 'id']


Phase_7_PCR/Train/train_candidates.json: reconstructing file:   0%|          |  0.00B /  160MB            

Phase_7_PCR/Train/train_candidates.json: downloading bytes:           |  0.00B            

fields: ['text', 'id']


... (6 outputs left out)

fields: ['text', 'id']


Phase_7_PCR/Test/test_candidates.json: reconstructing file:   0%|          |  0.00B / 71.1MB            

Phase_7_PCR/Test/test_candidates.json: downloading bytes:           |  0.00B            

fields: ['text', 'id']
{'Train': (827, 4320), 'Dev': (118, 1023), 'Test': (237, 1727)} (queries, candidates)


In [5]:
# Lexical scores for every query and candidate
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from scipy.sparse import csr_matrix
# It takes qs (queries) and cs (candidates) as input, computes BM25 and TF-IDF scores and returns (BM25 matrix, TF-IDF matrix).
def lexical(qs, cs):
    tv = TfidfVectorizer(sublinear_tf=True, ngram_range=(1, 2), min_df=2, max_features=300000).fit(cs)
    tfidf = (tv.transform(qs) @ tv.transform(cs).T).toarray()
    cv = CountVectorizer(min_df=2).fit(cs); C = cv.transform(cs).tocoo(); n = C.shape[0]
    df, dl = np.bincount(C.col, minlength=C.shape[1]), np.bincount(C.row, weights=C.data, minlength=n)
    w = C.data * 2.2 / (C.data + 1.2 * (.25 + .75 * dl[C.row] / dl.mean())) * np.log(1 + (n - df[C.col] + .5) / (df[C.col] + .5))
    Q = cv.transform(qs); Q.data = np.log1p(Q.data)
    return (Q @ csr_matrix((w, (C.row, C.col)), shape=C.shape).T).toarray(), tfidf
Lx = {s: lexical([q[1] for q in qs], [d[1] for d in cs]) for s, (qs, cs) in D.items()}
lex = Lx

In [6]:
# Scoring metric (Micro-F1@7) and score mixing
z = lambda S: (S - S.mean(1, keepdims=True)) / (S.std(1, keepdims=True) + 1e-9)
# It takes S (score matrix), qs (queries) and cs (candidates) as input, gets the top 7 candidates for each query and returns indices of the top 7 candidates per query.
def top7(S, qs, cs):
    S = np.where(np.array([d[0] for d in cs])[None] == np.array([q[0] for q in qs])[:, None], -1e9, S); return np.argsort(-S, 1)[:, :K]
# It takes S (score matrix), qs (queries) and cs (candidates) as input, computes Micro-F1 at 7 and returns the Micro-F1@7.
def micro_f1(S, qs, cs):
    tp = sum(len({cs[j][0] for j in r} & set(q[2])) for r, q in zip(top7(S, qs, cs), qs)); p, r = tp / (K * len(qs)), tp / sum(len(q[2]) for q in qs); return 2 * p * r / (p + r + 1e-9)
# It takes S (score matrix), lam (strength) and f (fraction) as input, lowers the scores of candidates that match too many queries and returns a adjusted score matrix.
def hub(S, lam, f):
    return S - lam * np.sort(S, 0)[-max(1, int(f * len(S))):].mean(0)
# It takes S (score matrix), t (temperature) and it (number of rounds) as input, balances scores so each candidate fits few queries and returns a balanced score matrix.
def sink(S, t, it=3):
    P = np.exp((S - S.max()) / t) + 1e-30
    for _ in range(it): P /= P.sum(0, keepdims=True); P /= P.sum(1, keepdims=True)
    return np.log(P)
# It takes model, lex (BM25 and TF-IDF scores), a (model weight), b (BM25 weight) and post (post-processing setting) as input, mixes model, BM25 and TF-IDF scores and returns a mixed score matrix.
def blend(model, lex, a, b, post):
    S = a * z(model) + (1 - a) * (b * z(lex[0]) + (1 - b) * z(lex[1])); return hub(S, *post[1:]) if post[0] == "hub" else sink(S, post[1])

In [7]:
# Fine-tuning methods: LoRMA, adapter, full
class LoRMA(nn.Module):
    # It takes base (base linear layer), r (rank) and alpha (scale of the low-rank update) as input, creates the small LoRMA matrices around a linear layer and returns nothing.
    def __init__(s, base, r, alpha):
        super().__init__(); s.base, s.s = base, alpha / r; s.A = nn.Parameter(torch.randn(r, base.out_features) / math.sqrt(r)); s.B = nn.Parameter(torch.zeros(base.out_features, r))
    # It takes x (input tensor) as input, runs the linear layer with the LoRMA update and returns the output of the updated layer.
    def forward(s, x): y = F.linear(x, s.base.weight.to(x.dtype)); y = y + s.s * F.linear(F.linear(y, s.A.to(x.dtype)), s.B.to(x.dtype)); return y if s.base.bias is None else y + s.base.bias.to(x.dtype)
class Adapter(nn.Module):
    # It takes blk (model block), d (model size) and r (adapter size) as input, creates a small adapter that starts as no change and returns nothing.
    def __init__(s, blk, d, r): super().__init__(); s.blk, s.dn, s.up = blk, nn.Linear(d, r), nn.Linear(r, d); nn.init.zeros_(s.up.weight); nn.init.zeros_(s.up.bias)
    # It takes x (input tensor) as input, runs the block and adds the adapter output and returns output with the adapter added.
    def forward(s, x): x = s.blk(x); return x + s.up(F.gelu(s.dn(x)))
class Ck(nn.Module):
    # It takes b (block) as input, wraps a block to save GPU memory and returns nothing.
    def __init__(s, b): super().__init__(); s.b = b
    # It returns the attention layer of the wrapped block.
    @property
    def attn(s): return s.b.attn
    # It takes *a (any arguments) as input, runs the block while saving memory and returns block output.
    def forward(s, *a): return torch.utils.checkpoint.checkpoint(s.b, *a, use_reentrant=False) if s.training and torch.is_grad_enabled() else s.b(*a)
# It takes bb (pre-trained backbone) and tech (fine-tuning method) as input, prepares the model for the chosen fine-tuning method and returns the trainable weights.
def adapt(bb, tech):
    lm = bb.model
    if tech != "full":
        for p in lm.parameters(): p.requires_grad_(False)
    if tech == "lorma":
        for n in bb.linear_names():
            par = lm; *path, leaf = n.split(".")
            for q in path: par = getattr(par, q)
            setattr(par, leaf, LoRMA(getattr(par, leaf), c["r"], c["alpha"]))
    if tech == "adapter":
        for i, b in enumerate(lm.blocks):
            if type(b).__name__ == "FeedForward": lm.blocks[i] = Adapter(b, bb.d_model, c["ar"])
        for p in lm.norm_f.parameters(): p.requires_grad_(True)
    if hasattr(bb, "ck"): bb.ck = True
    else: lm.blocks = nn.ModuleList(Ck(b) for b in lm.blocks)
    return [p for p in lm.parameters() if p.requires_grad]

In [8]:
# Convert documents to tokens and batches
# It takes texts as input, tokenizes documents, keeping the first part and returns a list of token arrays.
def tokenize(texts):
    n = c["chunk"] * c["kinf"]; return [np.array(tok.encode(t[:n * 6])[:n] or [0], np.int32) for t in texts]
# It takes rows (token lists) as input, pads token arrays and makes a mask and returns (padded ids, mask).
def pad(rows):
    L = torch.tensor([len(r) for r in rows]); x = torch.full((len(rows), int(L.max())), pid, dtype=torch.long)
    for i, r in enumerate(rows): x[i, :len(r)] = torch.as_tensor(r, dtype=torch.long)
    return x.to(DEV), (torch.arange(x.size(1))[None] < L[:, None]).to(DEV)

In [9]:
# Turn documents into vectors and compare them
# It takes docs (token arrays of the documents) and rng (random generator) as input, turns documents into vectors and returns one vector per document.
def doc_vectors(docs, rng=None):
    segs = []
    for i, d in enumerate(docs):
        n = max(1, math.ceil(len(d) / c["chunk"])); js = rng.choice(n, min(c["ktrain"], n), replace=False) if rng is not None else range(n)
        segs += [(i, d[j * c["chunk"]:(j + 1) * c["chunk"]]) for j in js]
    x, m = pad([s for _, s in segs]); ix = torch.tensor([i for i, _ in segs], device=DEV)
    with torch.autocast("cuda", dtype=torch.float16, enabled=DEV == "cuda"):
        h = bb.hidden(x, m); w = m[..., None].to(h.dtype); h = ((h * w).sum(1) / w.sum(1).clamp(min=1)).float()
    return torch.zeros(len(docs), h.size(1), device=DEV).index_add(0, ix, h) / torch.bincount(ix, minlength=len(docs))[:, None]
# It takes Xq (query token ids), Xc (candidate token ids) and bs (batch size) as input, returns a query x candidate similarity matrix; it is the cosine similarity between all queries and candidates.
@torch.no_grad()
def scores(Xq, Xc, bs=24):
    bb.model.eval(); vec = lambda X: torch.cat([doc_vectors(X[i:i + bs]).cpu() for i in range(0, len(X), bs)])
    q, k = vec(Xq), vec(Xc); mu = k.mean(0); return (F.normalize(q - mu, dim=-1) @ F.normalize(k - mu, dim=-1).T).numpy()

In [10]:
# Training loss (contrastive + distillation)
# It takes qi (query indices), di (candidate indices) and rng (random generator) as input, returns a loss value; it is the contrastive loss, plus distillation if a teacher is given.
def batch_loss(qi, di, rng):
    B = len(qi); v = doc_vectors([Xq[i] for i in qi] + [Xc[j] for j in di], rng); v = F.normalize(v - v.mean(0), dim=-1); lg = v[:B] @ v[B:].T / c["tau"]
    lg = lg - 1e4 * torch.tensor([[cj in gold[q] and k != i for k, cj in enumerate(di)] for i, q in enumerate(qi)], device=DEV)
    loss = F.cross_entropy(lg, torch.arange(B, device=DEV))
    if T is not None:
        t = torch.tensor(np.array([[T[q, j] for j in di] for q in qi]), device=DEV) / c["tau"] - (lg < -5e3) * 1e4
        loss = (1 - c["beta"]) * loss + c["beta"] * F.kl_div(F.log_softmax(lg, 1), F.softmax(t, 1), reduction="batchmean")
    return loss

In [11]:
# Training loop
# It takes ps (parameters) and evaluate (evaluation function) as input, trains the model and keeps the best epoch and returns training time in seconds.
def fit(ps, evaluate):
    rng = np.random.default_rng(c["seed"]); opt = torch.optim.AdamW(ps, lr=c["lr"], weight_decay=0.01); sc = torch.amp.GradScaler("cuda", enabled=DEV == "cuda")
    spe = len(pairs) // c["bs"]; total, t0 = c["epochs"] * spe, time.time(); best, bst = evaluate(), {k: p.detach().cpu().clone() for k, p in bb.model.named_parameters() if p.requires_grad}; print(f"   epoch 0 (no fine-tuning) dev {best:.4f}"); bb.model.train()
    for step in range(1, total + 1):
        for g in opt.param_groups: g["lr"] = c["lr"] * min(1, step / 20) * 0.5 * (1 + math.cos(math.pi * step / total))
        b = [pairs[i] for i in rng.choice(len(pairs), c["bs"], replace=False)]; qi = [x[0] for x in b]; di = [x[1] for x in b] + [int(rng.choice(hard[q])) for q in qi]
        opt.zero_grad(set_to_none=True); loss = batch_loss(qi, di, rng); sc.scale(loss).backward(); sc.unscale_(opt); nn.utils.clip_grad_norm_(ps, 1.0); sc.step(opt); sc.update()
        if step % 20 == 0: print(f"step {step}/{total} loss {loss.item():.4f} | {(time.time() - t0) / 60:.1f} min", flush=True)
        if step % spe == 0 or time.time() - t0 > c["hours"] * 3600:
            v = evaluate(); print(f"   dev micro-F1@7 (model + lexical + hubness) {v:.4f}", flush=True); bb.model.train()
            if v > best: best, bst = v, {k: p.detach().cpu().clone() for k, p in bb.model.named_parameters() if p.requires_grad}
            if time.time() - t0 > c["hours"] * 3600: break
    bb.model.load_state_dict(bst, strict=False); return time.time() - t0

In [12]:
# Prepare training pairs and hard negatives
X = {s: (tokenize([q[1] for q in qs]), tokenize([d[1] for d in cs])) for s, (qs, cs) in D.items()}
(qs, cs), (Xq, Xc) = D["Train"], X["Train"]; ix = {d[0]: j for j, d in enumerate(cs)}
gold = [{ix[r] for r in q[2] if r in ix} for q in qs]; pairs = [(i, j) for i, g in enumerate(gold) for j in g]
lx = z(Lx["Train"][0]) + z(Lx["Train"][1]); hard = [[j for j in np.argsort(-lx[i])[:c["hard"] + len(gold[i])] if j not in gold[i]] for i in range(len(qs))]
T = np.load(pick(c["teacher"]))["train_model"] if c["teacher"] else None; assert T is None or T.shape == (len(qs), len(cs)), "teacher scores do not match the training data"
print(f"{len(qs)} train queries, {len(pairs)} positive pairs, distillation: {T is not None}")

827 train queries, 5308 positive pairs, distillation: False


In [13]:
# Fine-tune the model
dq, dc = D["Dev"]; ps = adapt(bb, TECH); bb.model.to(DEV); ntr, nall = sum(p.numel() for p in ps), sum(p.numel() for p in bb.model.parameters())
print(f"{TECH}: {ntr / 1e6:.2f}M trainable of {nall / 1e6:.1f}M parameters")
ref = lambda: micro_f1(blend(scores(*X["Dev"]), lex["Dev"], .5, .5, ("hub", 1, .08)), dq, dc)
f1_before = micro_f1(scores(*X["Dev"]), dq, dc); print(f"dev micro-F1@7 with the Phase-1 representations only (raw): {f1_before:.4f}")
secs = fit(ps, ref)

adapter: 1.59M trainable of 211.8M parameters
   epoch 0 (no fine-tuning) dev 0.3902
step 20/1326 loss 4.1187 | 1.7 min
step 40/1326 loss 3.1010 | 2.4 min
step 60/1326 loss 4.1559 | 3.2 min
step 80/1326 loss 3.0830 | 3.9 min
step 100/1326 loss 2.5071 | 4.6 min
step 120/1326 loss 2.0392 | 5.3 min
step 140/1326 loss 2.2608 | 6.0 min
step 160/1326 loss 4.7266 | 6.7 min
step 180/1326 loss 2.9076 | 7.4 min
step 200/1326 loss 3.9850 | 8.1 min
step 220/1326 loss 2.1682 | 8.8 min
step 240/1326 loss 2.6331 | 9.5 min
step 740/1326 loss 1.5282 | 28.2 min
step 760/1326 loss 1.3728 | 28.9 min
step 780/1326 loss 3.5016 | 29.6 min
step 800/1326 loss 1.4950 | 30.3 min
step 820/1326 loss 1.7065 | 31.0 min
step 840/1326 loss 1.9558 | 31.7 min
step 860/1326 loss 2.2993 | 32.4 min
step 880/1326 loss 2.0529 | 33.1 min
step 900/1326 loss 2.4974 | 33.8 min
step 920/1326 loss 1.3362 | 34.5 min
step 940/1326 loss 1.7984 | 35.2 min
step 960/1326 loss 1.3504 | 35.9 min
step 980/1326 loss 2.2489 | 36.6 min
step 1

In [14]:
# Compute and save scores for all splits
S = {s: scores(*X[s]) for s in ("Dev", "Test")}
if c["teach"]: S["Train"] = scores(Xq, Xc)
np.savez_compressed(f"{OUT}/{NAME}_scores.npz", **{f"{s.lower()}_model": v for s, v in S.items()}, **{f"{s.lower()}_{k}": Lx[s][i] for s in Lx for i, k in enumerate(("bm25", "tfidf"))},
                    **{f"{s.lower()}_{k}": np.array([d[0] for d in D[s][i]]) for s in D for i, k in enumerate(("qids", "cids"))})

In [15]:
# Pick best settings on dev and write submissions
posts = [("hub", l, k) for l in (0, .5, 1, 1.5, 2, 3) for k in (.02, .08, .2)] + [("sink", t) for t in (.2, .35, .5, 1)]
res = {(a_, b_, p): micro_f1(blend(S["Dev"], lex["Dev"], a_, b_, p), dq, dc) for a_ in (1, .7, .5, .35, .25, .15, .08, 0) for b_ in (.5, 1) for p in posts}
f1_lex = micro_f1(z(lex["Dev"][0]) + z(lex["Dev"][1]), dq, dc); f1_raw = micro_f1(S["Dev"], dq, dc); print(f"dev micro-F1@7 | lexical only {f1_lex:.4f} | model only {f1_raw:.4f} | model before fine-tuning {f1_before:.4f}")
tq, tc = D["Test"]; top = sorted(res, key=res.get, reverse=True); print("best settings:", [(k, round(res[k], 4)) for k in top[:5]])
for a_, b_, p in top[:4]:
    pred = {q[0]: [tc[j][0] for j in r] for q, r in zip(tq, top7(blend(S["Test"], lex["Test"], a_, b_, p), tq, tc))}
    assert len(pred) == len(tq) and all(len(set(v)) == K for v in pred.values())
    f = f"{OUT}/{NAME}_dev{res[(a_, b_, p)]:.4f}_m{a_}_b{b_}_{p[0]}{'_'.join(map(str, p[1:]))}_submission.zip"; zipfile.ZipFile(f, "w", zipfile.ZIP_DEFLATED).writestr("prediction.json", json.dumps(pred)); print(f)
json.dump(dict(name=NAME, technique=TECH, trainable_M=ntr / 1e6, total_M=nall / 1e6, train_seconds=secs, dev_model_only=f1_raw, dev_before_ft=f1_before, dev_lexical=f1_lex, grid={str(k): v for k, v in res.items()}, settings=c), open(f"{OUT}/{NAME}_summary.json", "w"), indent=1)

dev micro-F1@7 | lexical only 0.3180 | model only 0.2857 | model before fine-tuning 0.2335
best settings: [((0, 0.5, ('hub', 1, 0.08)), 0.448), ((0.08, 0.5, ('hub', 1, 0.08)), 0.4436), ((0, 0.5, ('sink', 0.5)), 0.4436), ((0, 0.5, ('sink', 0.35)), 0.4425), ((0, 0.5, ('hub', 1, 0.2)), 0.4414)]
/kaggle/working/p7_gp_adapter_dev0.4480_m0_b0.5_hub1_0.08_submission.zip
/kaggle/working/p7_gp_adapter_dev0.4436_m0.08_b0.5_hub1_0.08_submission.zip
/kaggle/working/p7_gp_adapter_dev0.4436_m0_b0.5_sink0.5_submission.zip
/kaggle/working/p7_gp_adapter_dev0.4425_m0_b0.5_sink0.35_submission.zip


In [16]:
# Zip all outputs
with zipfile.ZipFile(f"{OUT}/ALL_outputs.zip", "w", zipfile.ZIP_DEFLATED) as zf:
    for p in sorted(glob.glob(f"{OUT}/*")):
        if os.path.isfile(p) and not p.endswith("ALL_outputs.zip"): zf.write(p, os.path.basename(p))